# Model Deployment on Kubernetes | 在 Kubernetes 上部署模型

In this demo we are going to look at different ways to run Model Deployments on Kubernetes.

在本演示中，我们将介绍在 Kubernetes 上运行模型部署的不同方法。

https://github.com/kubernetes-sigs/metrics-server/issues/196
For HPA example (metric server install):

## Having a Look around | 查看周围

In the `manifests` directory.

在 `manifests` 目录中。

```bash
manifests
├── deployment.yaml
├── hpa.yaml
├── service.yaml

```部署它！


Deploy it!

## Horizontal Pod Autoscaler | 水平 Pod 自动缩放器

HPA plays a huge role in being able to handle scaling your applications up and down.

HPA 在能够处理应用程序的纵向扩展和缩小方面发挥着巨大作用。

This is especially useful for ML/AI workloads.

这对于 ML/AI 工作负载特别有用。

```yaml
apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: ml-model-hpa
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: ml-model-deployment
  minReplicas: 3
  maxReplicas: 10
  metrics:
  - type: Resource
    resource:
      name: cpu
      target:
        type: Utilization
        averageUtilization: 70
```


`scaleTargetRef` defines what we are going to target to scale.`metrics` 是我们定义要监视的目标资源以触发缩放事件的地方。



`scaleTargetRef` 定义了我们将要缩放的目标。`metrics` is where we define the target resource to watch in order to trigger scaling events. 



`minReplicas` and `maxRelicas` define the minimum and maximum number of pods to be running high or light load.`minReplicas` 和 `maxReplicas` 定义了在高负载或轻负载时运行的 pod 的最小和最大数量。


## Node Affinity | 节点亲和性

Node affinity ensures that pods are scheduled on specific nodes by matching labels on the nodes with rules defined in the pod's configuration. 

节点亲和性通过将节点上的标签与 pod 配置中定义的规则进行匹配，确保 pod 被调度到特定节点。

It provides control over where workloads run, enabling optimization for hardware or specialized requirements, such as scheduling ML workloads on GPU-enabled nodes.

它提供了对工作负载运行位置的控制，从而对硬件或专门需求进行优化，例如在支持 GPU 的节点上调度 ML 工作负载。


```yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: ml-model-deployment
spec:
  selector:
    matchLabels:
      app: ml-model
  template:
    metadata:
      labels:
        app: ml-model
    spec:
      affinity:
        nodeAffinity:
          requiredDuringSchedulingIgnoredDuringExecution:
            nodeSelectorTerms:
            - matchExpressions:
              - key: kubernetes.io/hostname
                operator: In
                values:
                - "node02"
      containers:
      - name: ml-model-container
        image: wbassler/mobilenetv3lg-flask:v1.0
        imagePullPolicy: Always
        resources:
          requests:
            cpu: "200m"

            memory: "250Mi"```

          limits:            memory: "250Mi"
            cpu: "200m"

## Node Selector | 节点选择器

Another way to schedule pods to specific nodes is through the use of `nodeSelector`. 

将 pod 调度到特定节点的另一种方法是使用 `nodeSelector`。

`nodeSelector` is not a flexible and is a simple key value match whereas node affinity is more advanced and flexible due to rules requirements. 

`nodeSelector` 不灵活，是一个简单的键值匹配，而节点亲和性由于规则要求更加高级和灵活。

```yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: ml-model-deployment
spec:
  selector:
    matchLabels:
      app: ml-model
  template:
    metadata:
      labels:
        app: ml-model
    spec:
      nodeSelector:
        kubernetes.io/hostname: "node02"
      containers:
      - name: ml-model-container
        image: wbassler/mobilenetv3lg-flask:v1.0
        imagePullPolicy: Always
        resources:
          requests:
            cpu: "200m"

            memory: "250Mi"```

          limits:            memory: "250Mi"
            cpu: "200m"

## Taints | 污点

Taints prevent pods from being scheduled on specific nodes unless the pods explicitly tolerate the taint. 

污点防止 pod 被调度到特定节点，除非 pod 明确容忍该污点。

They are used to reserve nodes for specialized workloads or to isolate certain nodes.

它们用于为专门工作负载保留节点或隔离某些节点。

```bash
kubectl taint nodes node02 role=pytorch:NoSchedule

``````

kubectl describe node node02 | grep Taints
```bash

## Tolerations | 容忍

```yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: ml-model-deployment
spec:
  selector:
    matchLabels:
      app: ml-model
  template:
    metadata:
      labels:
        app: ml-model
    spec:
      tolerations:
      - key: "role"
        operator: "Equal"
        value: "pytorch"
        effect: "NoSchedule"
      nodeSelector:
        kubernetes.io/hostname: "node02"
      containers:
      - name: ml-model-container
        image: wbassler/mobilenetv3lg-flask:v1.0
        imagePullPolicy: Always
        resources:
          requests:
            cpu: "200m"
            memory: "250Mi"
          limits:
            cpu: "200m"
            memory: "250Mi"
```


## Assigning GPUs | 分配 GPU

Using GPU enabled nodes requires a couple additional things configured on your nodes. 

使用支持 GPU 的节点需要在您的节点上配置几个额外的东西。

For NVIDIA:

对于 NVIDIA：

1) Cluster has GPU-enabled nodes and that the NVIDIA drivers are installed on those nodes.

1) 集群具有支持 GPU 的节点，并且这些节点上已安装 NVIDIA 驱动程序。

2) NVIDIA device plugin installed. More information [https://github.com/NVIDIA/k8s-device-plugin](https://github.com/NVIDIA/k8s-device-plugin).

2) 已安装 NVIDIA 设备插件。更多信息请访问 [https://github.com/NVIDIA/k8s-device-plugin](https://github.com/NVIDIA/k8s-device-plugin)。

3) Update your deployment with below:

3) 使用以下内容更新您的部署：

```yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: ml-model-deployment
spec:
  selector:
    matchLabels:
      app: ml-model
  template:
    metadata:
      labels:
        app: ml-model
    spec:
      tolerations:
      - key: "role"
        operator: "Equal"
        value: "pytorch"
        effect: "NoSchedule"
      nodeSelector:
        kubernetes.io/hostname: "node02"
      containers:
      - name: ml-model-container
        image: wbassler/mobilenetv3lg-flask:v1.0
        imagePullPolicy: Always
        resources:
          requests:
            cpu: "200m"
            memory: "250Mi"
            nvidia.com/gpu: 1    # Request 1 GPU
          limits:
            cpu: "200m"
            memory: "250Mi"
            nvidia.com/gpu: 1    # Request 1 GPU
```

NOTE: Ensure that your application is configured to leverage GPU acceleration. ie: NVIDIA drivers and libraries

注意：确保您的应用程序配置为利用 GPU 加速。例如：NVIDIA 驱动程序和库